# ToE remote batch on Colab (inference frontier + GroupKAN)

Generates all remaining experimental data for the AAAI paper_v8 extension.

**Before running:** `Runtime > Change runtime type > GPU` (A100 or L4 if you have
Pro; T4 works but is ~3-5x slower).

**If the runtime disconnects at any point: just `Runtime > Run all` again.**
Every stage is resumable, and all results live on your Google Drive
(`MyDrive/ToE_outputs`), so completed work is never lost or redone.

Expected total GPU time: ~1-3 h on A100, ~2-6 h on L4, ~4-10 h on T4
(free-tier T4 may need 2-3 sessions; that is fine).

In [ ]:
# 1) GPU sanity
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: set Runtime > Change runtime type > GPU'

In [ ]:
# 2) Mount Drive (results persist here across disconnects)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# 3) Clone branch, point outputs/ at Drive, extract checkpoints, install deps
set -e
cd /content
[ -d ToE ] || git clone -b feature/hamiltonian-kan https://github.com/Hafez-Al-Khatib/ToE.git
mkdir -p /content/drive/MyDrive/ToE_outputs
cd ToE
if [ ! -L outputs ]; then rm -rf outputs; ln -s /content/drive/MyDrive/ToE_outputs outputs; fi
tar xzf remote_checkpoints.tar.gz   # pretrained checkpoints -> outputs/ (on Drive)
pip -q install lpips
echo SETUP_OK

In [ ]:
%%bash
# 4) THE BATCH (blocks for hours; live log below).
#    v1+v2 skip anything already done on Drive.
#    v3 (new): STL-10 64x64 frontier extension (pre-registered) +
#    the fixed oracle-gain stage picked up by v2's skip-if-done check.
cd /content/ToE
git pull
bash scripts/remote_run.sh
bash scripts/remote_run2.sh
bash scripts/remote_run3.sh
cp -f remote_run.log remote_run2.log remote_run3.log /content/drive/MyDrive/ToE_outputs/ || true

In [ ]:
# 5) Progress check (safe to run anytime, even mid-sweep)
import json, pathlib
parts = pathlib.Path('/content/ToE/outputs/inference_frontier/parts')
V2 = ('kan_32k_ts1', 'kan_32k_ts2', 'kan_110k_ts1', 'kan_110k_ts2',
      'ladder_1m', 'ladder_8m', 'ladder_30m')
v1 = v2 = 0
for p in sorted(parts.glob('*.json')):
    try:
        d = json.loads(p.read_text())
        if d.get('done'):
            if d.get('model') in V2:
                v2 += 1
            else:
                v1 += 1
    except Exception:
        pass
print(f'sweep: v1 {v1}/60, v2 {v2}/70 cells done')
checks = {
    'outputs/group_kan/group_kan_32k.pt': 'GroupKAN ckpt',
    'outputs/group_kan/fine_grid_group_kan_32k.json': 'fine-grid alpha',
    'outputs/inference_frontier/frontier_lpips.json': 'LPIPS',
    'outputs/replication/kan_110k_ts2.pt': 'replication training',
    'outputs/scale_ladder/ladder_30m.pt': 'scale ladder training',
    'outputs/inference_frontier/timing.json': 'wall-clock timing',
    'outputs/review_fixes/oracle_gain.json': 'oracle gain',
    'outputs/blur_law/blur_law.json': 'blur control',
}
for f, label in checks.items():
    ok = pathlib.Path('/content/ToE/' + f).exists()
    print(f'{label}:', 'OK' if ok else 'pending')

In [ ]:
%%bash
# 6) Package results (also kept on Drive as a backup)
cd /content/ToE
tar czhf results_back3.tar.gz outputs/inference_frontier outputs/group_kan \
  outputs/replication outputs/scale_ladder outputs/review_fixes \
  outputs/blur_law outputs/stl64 remote_run*.log
cp -f results_back3.tar.gz /content/drive/MyDrive/
ls -la results_back3.tar.gz

In [ ]:
# 7) Download to your machine (also in Drive root as backup)
from google.colab import files
files.download('/content/ToE/results_back3.tar.gz')

## Afterwards, on the Windows machine

Put `results_back3.tar.gz` at the repo root, then in PowerShell:

```powershell
tar xzf results_back3.tar.gz
```

Then tell Claude: **"batch v3 results are back"** — the STL-64 pre-registered
verdict, the oracle-gain number, and the final paper edits continue locally.